In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import BallTree
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
training_df = pd.read_csv(
    filepath_or_buffer='../data/training_faults_diagnostics.csv',
    low_memory=False
)
training_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1058069 entries, 0 to 1058068
Data columns (total 47 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   RecordID                   1058069 non-null  int64  
 1   EventTimeStamp             1058069 non-null  object 
 2   eventDescription           1002335 non-null  object 
 3   ecuSoftwareVersion         831493 non-null   object 
 4   ecuModel                   1002466 non-null  object 
 5   ecuMake                    1002466 non-null  object 
 6   ecuSource                  1058069 non-null  int64  
 7   spn                        1058069 non-null  int64  
 8   fmi                        1058069 non-null  int64  
 9   active                     1058069 non-null  bool   
 10  activeTransitionCount      1058069 non-null  int64  
 11  EquipmentID                1058069 non-null  object 
 12  MCTNumber                  1058069 non-null  int64  
 13  Latitude    

In [4]:
training_df.columns

Index(['RecordID', 'EventTimeStamp', 'eventDescription', 'ecuSoftwareVersion',
       'ecuModel', 'ecuMake', 'ecuSource', 'spn', 'fmi', 'active',
       'activeTransitionCount', 'EquipmentID', 'MCTNumber', 'Latitude',
       'Longitude', 'LocationTimeStamp', 'NearServiceStation', 'IsFullDerate',
       'Severity_Level', 'Severity_Level_Numeric', 'Derate_Target_4.0-2.0',
       'Derate_Target_8.0-2.0', 'Derate_Target_12.0-2.0', 'AcceleratorPedal',
       'BarometricPressure', 'CruiseControlActive', 'CruiseControlSetSpeed',
       'DistanceLtd', 'EngineCoolantTemperature', 'EngineLoad',
       'EngineOilPressure', 'EngineOilTemperature', 'EngineRpm',
       'EngineTimeLtd', 'FuelLevel', 'FuelLtd', 'FuelRate', 'FuelTemperature',
       'IgnStatus', 'IntakeManifoldTemperature', 'LampStatus', 'ParkingBrake',
       'ServiceDistance', 'Speed', 'SwitchedBatteryVoltage', 'Throttle',
       'TurboBoostPressure'],
      dtype='object')

In [5]:
unnecessary_columns = [
    'RecordID',
    'EventTimeStamp',
    'eventDescription',
    'ecuSoftwareVersion',
    'ecuMake',
    'ecuModel',
    'ecuSource',
    'activeTransitionCount',
    'EquipmentID',
    'MCTNumber',
    'Latitude',
    'Longitude',
    'LocationTimeStamp',
    'NearServiceStation',
    'IsFullDerate',
    'Severity_Level_Numeric',
    'Derate_Target_2.0-0.001',
    'Derate_Target_4.0-0.001',
    'Derate_Target_8.0-0.001',
    'AcceleratorPedal',
    'CruiseControlSetSpeed',
    'CruiseControlActive',
    'DistanceLtd',
    'EngineTimeLtd',
    'FuelLevel',
    'FuelLtd',
    'IgnStatus',
    'LampStatus',
    'ParkingBrake'
]
len(unnecessary_columns)

29

In [6]:
features = [
    'spn',
    'fmi',
    'active',
    'Severity_Level',
    'Derate_Target_12.0-0.001',
    'BarometricPressure',
    'EngineCoolantTemperature',
    'EngineLoad',
    'EngineOilPressure',
    'EngineOilTemperature',
    'EngineRpm',
    'FuelRate',
    'FuelTemperature',
    'IntakeManifoldTemperature',
    'ServiceDistance',
    'Speed',
    'SwitchedBatteryVoltage',
    'Throttle',
    'TurboBoostPressure'
]
len(features)

19

In [7]:
# Conver SPN and FMI values to strings
training_df['spn'] = training_df['spn'].astype(str)
training_df['fmi'] = training_df['fmi'].astype(str)

In [8]:
# Create dataset with desired features
training_df = training_df[features]
training_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1058069 entries, 0 to 1058068
Data columns (total 19 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   spn                        1058069 non-null  object 
 1   fmi                        1058069 non-null  object 
 2   active                     1058069 non-null  bool   
 3   Severity_Level             401510 non-null   object 
 4   Derate_Target_12.0-0.001   1058069 non-null  int64  
 5   BarometricPressure         521532 non-null   float64
 6   EngineCoolantTemperature   521591 non-null   float64
 7   EngineLoad                 521085 non-null   float64
 8   EngineOilPressure          521733 non-null   float64
 9   EngineOilTemperature       519690 non-null   float64
 10  EngineRpm                  522191 non-null   float64
 11  FuelRate                   520788 non-null   float64
 12  FuelTemperature            272924 non-null   float64
 13  IntakeManifo

## Identify features for imputing missing values

In [9]:
# Drop columns that have too many NaN values
nan_drop_threshold = 0.8

drop_nan_columns = training_df.columns[training_df.isna().mean() > nan_drop_threshold]
print(drop_nan_columns)

training_df = training_df.drop(columns=drop_nan_columns)

Index(['ServiceDistance', 'SwitchedBatteryVoltage'], dtype='object')


In [10]:
# Group categorical columns
categorical_columns = training_df.select_dtypes(include=["object", "bool"]).columns
print(categorical_columns)
print(len(categorical_columns))

# Group numeric columns
numeric_columns = training_df.select_dtypes(include=["int64", "float64"]).columns
print(numeric_columns)
print(len(numeric_columns))

Index(['spn', 'fmi', 'active', 'Severity_Level'], dtype='object')
4
Index(['Derate_Target_12.0-0.001', 'BarometricPressure',
       'EngineCoolantTemperature', 'EngineLoad', 'EngineOilPressure',
       'EngineOilTemperature', 'EngineRpm', 'FuelRate', 'FuelTemperature',
       'IntakeManifoldTemperature', 'Speed', 'Throttle', 'TurboBoostPressure'],
      dtype='object')
13


In [11]:
# Group numeric columns by threshold
nan_low_threshold = 0.4

low_nan_numeric_columns = training_df[numeric_columns].columns[
    training_df[numeric_columns].isna().mean() <= nan_low_threshold
]
print(low_nan_numeric_columns)
print(len(low_nan_numeric_columns))

medium_nan_numeric_columns = training_df[numeric_columns].columns[
    training_df[numeric_columns].isna().mean() > nan_low_threshold
]
print(medium_nan_numeric_columns)
print(len(medium_nan_numeric_columns))

Index(['Derate_Target_12.0-0.001'], dtype='object')
1
Index(['BarometricPressure', 'EngineCoolantTemperature', 'EngineLoad',
       'EngineOilPressure', 'EngineOilTemperature', 'EngineRpm', 'FuelRate',
       'FuelTemperature', 'IntakeManifoldTemperature', 'Speed', 'Throttle',
       'TurboBoostPressure'],
      dtype='object')
12


## Split training dataset

In [12]:
target = 'Derate_Target_12.0-0.001'

In [13]:
X = training_df.drop(columns=[target])
y = training_df[target]

In [14]:
y.isna().sum()

np.int64(0)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

## Create pipeline and fit model

In [16]:
categorical_pipe = Pipeline(
    steps=[
        ('categorical_imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]
)

low_nan_numeric_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('low_nan_numeric_imputer', SimpleImputer(strategy='median'))
    ]
)

medium_nan_numeric_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('medium_nan_numeric_imputer', IterativeImputer(max_iter=20, random_state=30))
    ]
)

In [17]:
ct = ColumnTransformer(
    transformers=[
        ('categorical_pipe', categorical_pipe, categorical_columns),
        ('low_nan_numeric_pipe', low_nan_numeric_pipe, low_nan_numeric_columns.drop('Derate_Target_12.0-0.001')),
        ('medium_nan_numeric_pipe', medium_nan_numeric_pipe, medium_nan_numeric_columns)
    ]
)

In [18]:
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('model', MLPClassifier(
            activation='relu',
            hidden_layer_sizes=(64,64,64)
        ))
    ]
)

In [19]:
pipe.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Pipeline(steps=[('transformer',
                 ColumnTransformer(transformers=[('categorical_pipe',
                                                  Pipeline(steps=[('categorical_imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['spn', 'fmi', 'active', 'Severity_Level'], dtype='object')),
                                                 ('low_nan_numeric_pipe',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler()),
                                                                  ('low_n...
                                                                  ('medium_nan_numeric_imputer',
                                                                   IterativeImputer(max_iter=20,
                                                                                    random_state=30))]),
                                                  Index(['BarometricPressure', 'EngineCoolantTemperature', 'EngineLoad',
       'EngineOilPressure', 'EngineOilTemperature', 'EngineRpm', 'FuelRate',
       'FuelTemperature', 'IntakeManifoldTemperature', 'Speed', 'Throttle',
       'TurboBoostPressure'],
      dtype='object'))])),
                ('model', MLPClassifier(hidden_layer_sizes=(64, 64, 64)))])

In [20]:
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

In [21]:
print(classification_report(
    y_true=y_train,
    y_pred=y_pred_train
))

print(classification_report(
    y_true=y_test,
    y_pred=y_pred_test
))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    739352
           1       0.80      0.16      0.27      1296

    accuracy                           1.00    740648
   macro avg       0.90      0.58      0.64    740648
weighted avg       1.00      1.00      1.00    740648

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    316865
           1       0.56      0.07      0.12       556

    accuracy                           1.00    317421
   macro avg       0.78      0.54      0.56    317421
weighted avg       1.00      1.00      1.00    317421



In [22]:
training_cm = confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train
)
print(training_cm)

training_test_cm = confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)
print(training_test_cm)

[[739298     54]
 [  1084    212]]
[[316834     31]
 [   517     39]]


In [23]:
print(f'Training Savings: {(training_cm[1][1]*4000) - (training_cm[1][0]*500)}')
print(f'Training Test Savings: {(training_test_cm[1][1]*4000) - (training_test_cm[1][0]*500)}')

Training Savings: 306000
Training Test Savings: -102500


In [24]:
testing_df = pd.read_csv(
    filepath_or_buffer='data/testing_faults_diagnostics.csv',
    low_memory=False
)

In [25]:
testing_target = testing_df[target]

In [26]:
testing_df = testing_df.drop(columns=unnecessary_columns)
testing_df = testing_df.drop(columns=drop_nan_columns)
testing_df = testing_df.drop(columns=target)
testing_df.columns

Index(['spn', 'fmi', 'active', 'Severity_Level', 'BarometricPressure',
       'EngineCoolantTemperature', 'EngineLoad', 'EngineOilPressure',
       'EngineOilTemperature', 'EngineRpm', 'FuelRate', 'FuelTemperature',
       'IntakeManifoldTemperature', 'Speed', 'Throttle', 'TurboBoostPressure'],
      dtype='object')

In [27]:
testing_df['prediction'] = pipe.predict(testing_df)

In [28]:
print(classification_report(
    y_true=testing_target,
    y_pred=testing_df['prediction']
))

              precision    recall  f1-score   support

           0       1.00      0.78      0.88    128979
           1       0.00      0.38      0.01       287

    accuracy                           0.78    129266
   macro avg       0.50      0.58      0.44    129266
weighted avg       1.00      0.78      0.88    129266



In [29]:
testing_cm = confusion_matrix(
    y_true=testing_target,
    y_pred=testing_df['prediction']
)
testing_cm

array([[100867,  28112],
       [   178,    109]])

In [30]:
print(f'Savings: {(testing_cm[1][1]*4000) - (testing_cm[1][0]*500)}')

Savings: 347000
